# AI Brochure Generator — Phase 1

**Goal:** Paste any company URL, get a marketing brochure (Markdown).

**Pipeline:**
1. Scrape the landing page (requests + BeautifulSoup)
2. Ask the LLM which links matter (About, Careers, Products) and which to skip (Terms, Privacy, social)
3. Scrape those pages too
4. Feed all content to the LLM and ask for a structured JSON brochure
5. Render the JSON as Markdown

**Why this design:** Most company sites have dozens of links. Naively scraping all of them is slow, wastes tokens, and dilutes the signal. Letting the LLM pre-filter links is cheap (one cheap call) and dramatically improves the final brochure quality.

In [21]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown, display

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Try these models in order. Gemini 2.5-flash sometimes returns 503 under load —
# we fall through to lite or "latest" alias as a safety net.
MODELS = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-flash-latest"]

def llm_json(prompt_parts):
    """Call Gemini with JSON mode, with model fallback and retry on 503."""
    last_err = None
    for model in MODELS:
        for attempt in range(3):
            try:
                r = client.models.generate_content(
                    model=model,
                    contents=list(prompt_parts),
                    config={"response_mime_type": "application/json"},
                )
                return json.loads(r.text)
            except Exception as e:
                last_err = e
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    time.sleep(2 + attempt * 2)  # backoff
                    continue
                break  # non-retryable: try next model
    raise last_err

## Step 1 — Scrape a page

The `Website` class fetches a URL and exposes:
- `title` — page `<title>` tag
- `text` — clean body text (script/style/nav stripped — these add noise without info)
- `links` — every absolute URL on the page (the LLM filters these in step 2)

We strip `<script>`, `<style>`, `<nav>`, `<footer>`, `<img>`, `<input>` because their text content is either noise (JS code, CSS) or repeated across pages (nav menus). Keeping them blows up token usage for no signal.

In [22]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
}

class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        self.title = soup.title.string.strip() if soup.title and soup.title.string else "No title"
        for tag in soup(["script", "style", "img", "input", "nav", "footer"]):
            tag.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
        self.links = list({
            urljoin(url, a.get("href"))
            for a in soup.find_all("a")
            if a.get("href")
        })

    def contents(self):
        return f"Page: {self.title}\nURL: {self.url}\n\n{self.text}"

In [23]:
# Quick sanity check
site = Website("https://anthropic.com")
print("Title:", site.title)
print("Links found:", len(site.links))
print("\nFirst 500 chars of text:")
print(site.text[:500])

Title: Home \ Anthropic
Links found: 15

First 500 chars of text:
Skip to main content
Skip to footer
AI
research
and
products
that put safety at the frontier
AI will have a vast impact on the world. Anthropic is a public benefit corporation dedicated to securing its benefits and mitigating its risks.
Project Glasswing
Securing critical software for the AI era
Continue reading
Read the story
Read the story
Latest releases
Claude Opus 4.8
An upgrade to Opus across coding, agentic tasks, and professional work, with the consistency to handle long-running work.
Mo


## Step 2 — Let the LLM pick relevant links

We hand the LLM the list of links and ask it to return ONLY the ones useful for a brochure. Two key prompt techniques:

1. **Few-shot by description**: telling it what types of pages to keep (About, Careers, Products) and what to skip (Terms, Privacy, social media) gives concrete categories without writing examples.
2. **Structured output via `response_mime_type: application/json`**: Gemini natively supports JSON mode — no string parsing, no "sometimes it adds ```json" garbage.

In [24]:
LINK_FILTER_PROMPT = """You are given a list of links found on a company website.
Decide which links are most relevant for building a marketing brochure about the company.

KEEP links like: About, Team, Careers, Products, Services, Mission, Customers, Case Studies.
SKIP links like: Terms of Service, Privacy, Login, Sign up, social media (twitter/linkedin/etc), \
individual blog posts, anchor links (#section), email/tel links.

Respond ONLY with JSON in this exact format:
{
  "links": [
    {"type": "about page", "url": "https://full.url/about"},
    {"type": "careers page", "url": "https://full.url/careers"}
  ]
}

Use full absolute URLs. Pick at most 5 links."""

def pick_links(website):
    user_prompt = (
        f"Website: {website.url}\n\n"
        "Links found on this page:\n" + "\n".join(website.links)
    )
    return llm_json([LINK_FILTER_PROMPT, user_prompt])

In [25]:
relevant = pick_links(site)
print(json.dumps(relevant, indent=2))

{
  "links": [
    {
      "type": "homepage",
      "url": "https://www.anthropic.com/"
    },
    {
      "type": "product overview",
      "url": "https://claude.com/product/overview"
    },
    {
      "type": "research page",
      "url": "https://www.anthropic.com/research"
    },
    {
      "type": "mission/principles page",
      "url": "https://www.anthropic.com/constitution"
    },
    {
      "type": "product page",
      "url": "https://www.anthropic.com/claude/opus"
    }
  ]
}


## Step 3 — Gather content from the chosen pages

Scrape each link the LLM picked, concatenate everything with section markers, and return a single blob. Section markers (`--- about page ---`) help the next LLM call understand the structure of what it's reading.

Errors are caught per-link — one broken page shouldn't kill the whole run.

In [26]:
def gather_content(root_url):
    site = Website(root_url)
    parts = [f"=== Landing page ===\n{site.contents()}"]
    picked = pick_links(site)
    for link in picked.get("links", []):
        try:
            page = Website(link["url"])
            parts.append(f"\n=== {link['type']} ===\n{page.contents()}")
        except Exception as e:
            print(f"Skipped {link['url']}: {e}")
    return "\n".join(parts)

## Step 4 — Generate the brochure (structured JSON)

We ask the LLM for a fixed JSON shape. This is the key design decision for phase 2 — when we wrap this in FastAPI, the frontend will rely on these field names being stable.

We truncate scraped content to 20k chars before sending. Most sites have far less, but a content-heavy site (lots of product pages) can balloon. Truncation keeps token usage predictable on the free tier.

In [27]:
BROCHURE_PROMPT = """You are a marketing copywriter. Given scraped content from a company's website, \
write a short, engaging brochure for prospective customers, investors, and recruits.

Keep it concrete — pull real facts from the scraped content. Avoid generic marketing fluff.

Respond ONLY with JSON in this exact format:
{
  "company_name": "...",
  "tagline": "one punchy sentence",
  "about": "2-3 sentences on what the company is and who it serves",
  "what_we_do": ["bullet", "bullet", "bullet"],
  "why_us": ["bullet", "bullet", "bullet"],
  "culture": "2-3 sentences on values, team, working style",
  "careers": "1-2 sentences on hiring or open roles if mentioned, else a general invitation",
  "call_to_action": "one sentence telling the reader what to do next"
}"""

def generate_brochure(url):
    content = gather_content(url)
    content = content[:20000]  # cap tokens
    return llm_json([BROCHURE_PROMPT, f"Scraped website content:\n\n{content}"])

## Step 5 — Render as Markdown

In [28]:
def to_markdown(b):
    md = f"# {b['company_name']}\n\n"
    md += f"*{b['tagline']}*\n\n"
    md += f"## About\n{b['about']}\n\n"
    md += "## What We Do\n" + "\n".join(f"- {x}" for x in b['what_we_do']) + "\n\n"
    md += "## Why Us\n" + "\n".join(f"- {x}" for x in b['why_us']) + "\n\n"
    md += f"## Culture\n{b['culture']}\n\n"
    md += f"## Careers\n{b['careers']}\n\n"
    md += f"**{b['call_to_action']}**\n"
    return md

## End-to-end test

Change the URL and re-run. If this works on 2–3 different company sites, phase 1 is done — move to phase 2 (FastAPI wrapper).

In [29]:
brochure = generate_brochure("https://anthropic.com")
print(json.dumps(brochure, indent=2))

{
  "company_name": "Anthropic",
  "tagline": "AI research and products that put safety at the frontier.",
  "about": "Anthropic is a public benefit corporation dedicated to securing the benefits of AI and mitigating its risks, building AI to serve humanity's long-term well-being. We serve business users, consumers, and developers with frontier intelligence.",
  "what_we_do": [
    "Develop advanced AI models like Claude Opus 4.8, our most capable model for professional work.",
    "Secure critical software for the AI era through initiatives like Project Glasswing.",
    "Offer Anthropic Academy, providing resources and courses for AI fluency, API development, and enterprise deployment.",
    "Power real-world applications, including the first AI-assisted drive of NASA\u2019s Perseverance rover on Mars."
  ],
  "why_us": [
    "We are a public benefit corporation prioritizing AI safety and responsible scaling.",
    "Our Claude models offer genuinely helpful conversations with no ads o

In [30]:
display(Markdown(to_markdown(brochure)))

# Anthropic

*AI research and products that put safety at the frontier.*

## About
Anthropic is a public benefit corporation dedicated to securing the benefits of AI and mitigating its risks, building AI to serve humanity's long-term well-being. We serve business users, consumers, and developers with frontier intelligence.

## What We Do
- Develop advanced AI models like Claude Opus 4.8, our most capable model for professional work.
- Secure critical software for the AI era through initiatives like Project Glasswing.
- Offer Anthropic Academy, providing resources and courses for AI fluency, API development, and enterprise deployment.
- Power real-world applications, including the first AI-assisted drive of NASA’s Perseverance rover on Mars.

## Why Us
- We are a public benefit corporation prioritizing AI safety and responsible scaling.
- Our Claude models offer genuinely helpful conversations with no ads or sponsored content.
- Claude Opus 4.8 provides frontier intelligence for advanced coding, complex agentic workflows, and high-stakes enterprise tasks.
- Experience adaptive thinking, where our models adjust effort based on task complexity, delivering consistent reliability and precision.

## Culture
Our culture is rooted in core views on AI safety and a responsible scaling policy, ensuring that we build AI to serve humanity's long-term well-being. We foster a collaborative environment focused on alignment science and empowering users to build and learn with Claude.

## Careers
While we didn't mention specific roles, we invite you to explore how your talent can contribute to building safe, beneficial AI for humanity.

**Visit anthropic.com to try Claude, get API access, or learn more about our commitment to responsible AI.**
